## Датасет Credit Card Fraud Detection.
Практический проект по построению системы поиска аномалий.

### Часть 0. Подготовка

In [26]:
# Инициализация и логирование

debug = 1  # 1 — выводить отладочные логи, 0 — скрывать

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # Защита от падения ядра Jupyter

import cv2
import torch
import platform
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Скачиваем архив напрямую в Colab по ID файла из вашей ссылки
!gdown "1leoj71B_D_paBLC1uP5EKcnR5lJFPovn" -O /content/understanding_cloud_organization.zip

# 2. Создаем чистую папку для датасета
!mkdir -p /content/cloud_data

# 3. Распаковываем архив в созданную папку
!unzip -q /content/understanding_cloud_organization.zip -d /content/cloud_data

# 4. Проверяем, что файлы на месте
!ls -l /content/cloud_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Downloading...
From (original): https://drive.google.com/uc?id=1leoj71B_D_paBLC1uP5EKcnR5lJFPovn
From (redirected): https://drive.google.com/uc?id=1leoj71B_D_paBLC1uP5EKcnR5lJFPovn&confirm=t&uuid=25b00ee7-ce82-4745-b866-c13ae962699d
To: /content/understanding_cloud_organization.zip
100% 6.23G/6.23G [05:29<00:00, 18.9MB/s]
total 209420
-rw-r--r-- 1 root root    329148 Dec 12  2019 sample_submission.csv
drwxr-xr-x 2 root root    131072 Sep 22 09:36 test_images
-rw-r--r-- 1 root root 213826644 Dec 12  2019 train.csv
drwxr-xr-x 2 root root    155648 Sep 22 09:37 train_images


In [27]:
# Полезные найтроки

def log(message):
    if debug == 1:
        # %Y-%m-%d — дата, %H:%M:%S — время
        timestamp = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
        print(f"[DEBUG] {timestamp} {message}")

log("Функция log готова. Использование: log(f\"message: {variable}\").")

np.random.seed(42)
num_cpu = os.cpu_count() or 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log(device)

[DEBUG] 2026-09-22T09:48:10 Функция log готова. Использование: log(f"message: {variable}").
[DEBUG] 2026-09-22T09:48:10 cuda


### Часть 1. Формирование нейронной сверточной сети

In [28]:
class CloudUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=4): # снимки цветные в RGB, так что каналов 3, категорий в датасете 4: (Fish, Flower, Gravel, Sugar)
        super(CloudUNet, self).__init__()

        # В архитектуре U-Net на каждом уровне разрешение картинки обрабатывается дважды.
        # Чтобы не дублировать код, создана мини-функция, генерирующая блок из двух сверток:
        def double_conv(in_c, out_c):
            return nn.Sequential(
            # ядро 3х3 - мировой стандарт: свертка смотрит на пиксель его соседей, для того чтобы обрабатывала граничные добавляем рамку:
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c), # Пакетная нормализация, слой масштабирует и центрирует значения пикселей (активаций) внутри сети, приводя их к среднему 0 и дисперсии 1. Без него будет тяжко.
                nn.SiLU(), # Swish вместо классического ReLU, гладкая в районе нуля и имеет небольшую область для отрицательных значений. Для природных объектов, таких как облака подходит лучше.
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.SiLU()
            )

        # Энкодер (Сжатие / Извлечение признаков)
        self.enc1 = double_conv(in_channels, 32) # тут сеть видит мелкие детали
        self.pool1 = nn.MaxPool2d(2) # уменьшение картинки в два раза окном 2х2 чтобы видела общие очертания
        self.enc2 = double_conv(32, 64) # увеличение кол-ва каналов
        self.pool2 = nn.MaxPool2d(2) # уменьшение картинки в два раза

        # Горлышко (Bottleneck)
        self.bottleneck = double_conv(64, 128) # картинка уменьшилась в 4 раза, границы пикселей стёрты, но смысл модель извлекла

        # Декодер (Расширение / Восстановление маски)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2) # увеличение в 2 раза
        self.dec2 = double_conv(128, 64) # 128 за счет skip-connection (64 + 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2) # увеличение в 2 раза
        self.dec1 = double_conv(64, 32) # 64 за счет skip-connection (32 + 32)

        # Финальный слой выдает маски для 4 классов облаков. Свертка 1х1 не смотрит на соседние пиксели. Её задача — просто взять 32 глубоких признака, скопившихся в каждом пикселе на финише, и математически спроецировать их в 4 финальных канала. На выходе получается матрица, имеющая в точности исходные ширину и высоту картинки, и состоящая из 4 слоев (по слою на каждый тип облака).
        self.final = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        # Шаг вниз
        s1 = self.enc1(x)   # Сохраняем s1 — это мост для верхнего уровня
        p1 = self.pool1(s1) # Уменьшаем размер в 2 раза
        s2 = self.enc2(p1)  # Сохраняем s2 — это мост для среднего уровня
        p2 = self.pool2(s2) # Уменьшаем размер еще в 2 раза

        # Центр
        b = self.bottleneck(p2) # Самые абстрактные признаки

        # Шаг вверх с конкатенацией (Skip-connections)
        u2 = self.up2(b)                 # Увеличили размер в 2 раза. Получили 64 канала.
        c2 = torch.cat([u2, s2], dim=1)  # Склеиваем с s2 (тоже 64 канала). Итого 128.
        d2 = self.dec2(c2)               # Прогнали через свертки, сжали в 64 канала

        u1 = self.up1(d2)                # Увеличили размер до исходного. Получили 32 канала.
        c1 = torch.cat([u1, s1], dim=1)  # Склеиваем с s1 по первому каналу (dim=1). Итого 64.
        d1 = self.dec1(c1)               # Прогнали через финальные свертки, получили 32 канала.

        return self.final(d1)            # Превращаем в 4 карты предсказаний.
log("Готово")

[DEBUG] 2026-09-22T09:48:13 Готово


### Часть 2. Обучение Сети на представленных данных и оценка её качетва с помощью Dice

In [37]:
# Остановка в случае если качество начало паадть
class EarlyStopping:
    def __init__(self, patience=3, verbose=True, delta=0.001):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_dice, model):
        score = val_dice
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'Ранний останов: счетчик {self.counter} из {self.patience} эпох без улучшений.')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), 'best_cloud_unet.pth')

# Метрика Дайса
def dice_coefficient(preds, targets, smooth=1e-6):
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).float()
    targets = targets.float()

    # ИСПРАВЛЕНО: Заменили .view на .reshape, чтобы избежать ошибок stride на GPU
    preds_flat = preds.reshape(-1)
    targets_flat = targets.reshape(-1)

    intersection = (preds_flat * targets_flat).sum()
    total_area = preds_flat.sum() + targets_flat.sum()

    return (2. * intersection + smooth) / (total_area + smooth)

class OptimizedBCEDiceLoss(nn.Module):
    def __init__(self):
        super(OptimizedBCEDiceLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)

        probs = torch.sigmoid(preds)
        probs_flat = probs.reshape(probs.size(0), -1)
        targets_flat = targets.reshape(targets.size(0), -1)

        inter = (probs_flat * targets_flat).sum(dim=1)
        den = probs_flat.sum(dim=1) + targets_flat.sum(dim=1)

        dice_loss = 1.0 - (2.0 * inter + 1e-6) / (den + 1e-6)

        # ИСПРАВЛЕНО: Увеличиваем приоритет Dice компоненты в 3 раза, чтобы избежать коллапса в ноль, при первых прогонах артефакты из-за пустотсы в датасете
        return bce_loss + 3.0 * dice_loss.mean()

# Оптимизация декодера
def rle_decode(mask_rle, shape=(1400, 2100)):
    if pd.isna(mask_rle) or mask_rle == '':
        return np.zeros(shape, dtype=np.float32)
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths

    # перемножаем высоту на ширину, получая число (инт), а не кортеж
    img = np.zeros(shape[0] * shape[1], dtype=np.float32)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1.0
    return img.reshape(shape)

# Работа с датасетом
class CloudDataset(Dataset):
    def __init__(self, csv_path, img_dir, img_size=(128, 192)):
        self.img_dir = img_dir
        self.img_size = img_size # формат (высота, ширина)

        # Читаем CSV разметки
        df = pd.read_csv(csv_path)

        # В Kaggle строки склеены: '002be4f.jpg_Fish'. Разделяем их на имя картинки и класс
        df[['ImageId', 'Label']] = df['Image_Label'].str.split('_', expand=True)

        self.classes = ['Fish', 'Flower', 'Gravel', 'Sugar']
        self.img_ids = df['ImageId'].unique()
        self.df_indexed = df.set_index(['ImageId', 'Label'])

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = os.path.join(self.img_dir, img_id)

        # 1. Чтение и ресайз изображения
        image = cv2.imread(img_path)
        if image is None:
            # На случай, если какой-то файл поврежден, отдаем пустой тензор
            return torch.zeros((3, self.img_size[0], self.img_size[1])), torch.zeros((4, self.img_size[0], self.img_size[1]))

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Обратите внимание: cv2.resize принимает (ширина, высота)
        image = cv2.resize(image, (self.img_size[1], self.img_size[0]))
        image = image.transpose(2, 0, 1) / 255.0 # в формат [Channels, Height, Width]

        # 2. Сборка маски из 4 каналов
        mask = np.zeros((4, self.img_size[0], self.img_size[1]), dtype=np.float32)
        for i, cls in enumerate(self.classes):
            try:
                rle_string = self.df_indexed.loc[(img_id, cls), 'EncodedPixels']
                if pd.notna(rle_string):
                    # Декодируем в полный размер, затем сжимаем методом ближайшего соседа (чтобы сохранить 0 и 1)
                    full_mask = rle_decode(rle_string)
                    mask[i] = cv2.resize(full_mask, (self.img_size[1], self.img_size[0]), interpolation=cv2.INTER_NEAREST)
            except KeyError:
                pass # Если класса нет для этой картинки, оставляем нули

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

# Для выводв визуальной информации
def visualize_live_prediction(epoch, val_dataset, model, device):
    model.eval()

    # Явно берем самую первую картинку и её маску по индексу [0]
    img_tensor, true_mask_tensor = val_dataset[0]

    # Добавляем размерность батча [1, 3, H, W] и отправляем на GPU
    input_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            pred_logits = model(input_tensor)
            # Применяем сигмоиду и бинаризуем по порогу 0.5, как в метрике Дайса
            pred_probs = torch.sigmoid(pred_logits)
            pred_mask = (pred_probs > 0.5).float().cpu().squeeze().numpy()

    # Переводим исходную картинку обратно в [H, W, Channels] для imshow
    original_img = img_tensor.numpy().transpose(1, 2, 0)

    # Суммируем маски по всем 4 каналам классов для наглядности
    true_mask_combined = true_mask_tensor.numpy().sum(axis=0)
    pred_mask_combined = pred_mask.sum(axis=0) if pred_mask.ndim == 3 else pred_mask

    # Отрисовка
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(original_img)
    axes[0].set_title(f"Эпоха {epoch} | Снимок с орбиты")

    axes[1].imshow(original_img)
    if true_mask_combined.sum() > 0:
        axes[1].imshow(true_mask_combined, cmap='jet', alpha=0.4)
    axes[1].set_title("Реальные облака (Kaggle)")

    axes[2].imshow(original_img)
    if pred_mask_combined.sum() > 0:
        axes[2].imshow(pred_mask_combined, cmap='jet', alpha=0.4)
    axes[2].set_title("Предсказание U-Net")

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

log("Функции загружены")

[DEBUG] 2026-09-22T10:19:56 Функции загружены


In [39]:
# Инициализация модели
model = CloudUNet(in_channels=3, out_channels=4).to(device)

criterion = OptimizedBCEDiceLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

log("Модель инициилизирована")

[DEBUG] 2026-09-22T10:37:25 Модель инициилизирована


In [31]:
!ls -F /content/sample_data/

anscombe.json*		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md*


In [41]:
# Настройка загрузчика
# В Dataset уменьшаем размер изображений, для процессора это критически важно: уменьшение размера в 2 раза ускоряет расчеты в 4 раза!
# BASE_DIR = "understanding_cloud_organization"
BASE_DIR = "/content/cloud_data"
CSV_PATH = os.path.join(BASE_DIR, "train.csv")
IMG_DIR = os.path.join(BASE_DIR, "train_images")

full_dataset = CloudDataset(csv_path=CSV_PATH, img_dir=IMG_DIR, img_size=(128, 192))

train_len = int(0.85 * len(full_dataset))
val_len = len(full_dataset) - train_len
train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

log(f"Всего снимков в базе: {len(full_dataset)}")
log(f"Размер обучающей выборки: {len(train_dataset)}")
log(f"Размер валидационной выборки: {len(val_dataset)}")

[DEBUG] 2026-09-22T10:49:29 Всего снимков в базе: 5546
[DEBUG] 2026-09-22T10:49:29 Размер обучающей выборки: 4714
[DEBUG] 2026-09-22T10:49:29 Размер валидационной выборки: 832


In [42]:
# Цикл обучения с валидацией и оценкой
epochs = 10
early_stopping = EarlyStopping(patience=3, verbose=True)
history = {'train_loss': [], 'val_loss': [], 'val_dice': []}

print(f"Старт обучения. Потоков задействовано: {torch.get_num_threads()}")
for epoch in range(1, epochs + 1):
    # --- ОБУЧЕНИЕ ---
    model.train()
    running_train_loss = 0.0

    # nrows=2 аккуратно обновляет одну и ту же строчку в консоли, не забивая экран ноутбука
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch:02d}/{epochs} [Train]", leave=False, nrows=2)

    for imgs, masks in pbar:
        imgs = imgs.to(device)
        masks = masks.to(device)

        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(imgs)
            loss = criterion(outputs, masks)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * imgs.size(0)

        # Динамически обновляем текущий лосс прямо в строке
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    epoch_train_loss = running_train_loss / len(train_dataset)

    # --- ВАЛИДАЦИЯ ---
    model.eval()
    running_val_loss = 0.0
    running_val_dice = 0.0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device, memory_format=torch.channels_last)
            masks = masks.to(device)

            with torch.amp.autocast(device_type='cpu', dtype=torch.bfloat16):
                outputs = model(imgs)
                loss = criterion(outputs, masks)

            running_val_loss += loss.item() * imgs.size(0)
            running_val_dice += dice_coefficient(outputs, masks).item() * imgs.size(0)

    epoch_val_loss = running_val_loss / len(val_dataset)
    epoch_val_dice = running_val_dice / len(val_dataset)

    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_dice'].append(epoch_val_dice)

    log(f"Итог эпохи {epoch:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Dice: {epoch_val_dice:.4f}")

    # Каждые 2 эпохи выводим визуальный тест предсказания
    if epoch % 2 == 0 or epoch == 1:
        visualize_live_prediction(epoch, val_dataset, model, device)

    # Проверка условия ранней остановки
    early_stopping(epoch_val_dice, model)
    if early_stopping.early_stop:
        print("Ранний останов.")
        break

Output hidden; open in https://colab.research.google.com to view.